# Reasoning dan Long Context Demo

Notebook ini merangkum `02-reasoning-dan-long-context.md`: perbandingan gaya prompting, estimasi biaya konteks, lost in the middle, dan alasan praktis memilih RAG.

In [ ]:
from collections import Counter
import textwrap
import matplotlib.pyplot as plt

## API LLM Opsional

Notebook ini tetap bisa jalan tanpa API key. Jika ingin memanggil contoh LLM, isi `GEMINI_API_KEY` melalui Kaggle Secrets atau environment variable. API key bisa dibuat dari Google AI Studio; free tier memiliki kuota dan dapat berubah.

In [ ]:
import os
import json
import urllib.request
import urllib.error

def get_secret(name):
    try:
        from kaggle_secrets import UserSecretsClient
        value = UserSecretsClient().get_secret(name)
        if value:
            return value
    except Exception:
        pass
    return os.getenv(name, "")

GEMINI_API_KEY = get_secret("GEMINI_API_KEY") or "PASTE_GEMINI_API_KEY_HERE"
GEMINI_MODEL = os.getenv("GEMINI_MODEL", "gemini-2.5-flash")

def call_gemini_text(prompt, model=GEMINI_MODEL):
    if not GEMINI_API_KEY or GEMINI_API_KEY.startswith("PASTE_"):
        return "[SKIP] Isi GEMINI_API_KEY di Kaggle Secrets atau environment variable untuk memanggil LLM."

    url = f"https://generativelanguage.googleapis.com/v1beta/models/{model}:generateContent"
    payload = {"contents": [{"parts": [{"text": prompt}]}]}
    request = urllib.request.Request(
        url,
        data=json.dumps(payload).encode("utf-8"),
        headers={"Content-Type": "application/json", "x-goog-api-key": GEMINI_API_KEY},
        method="POST",
    )

    try:
        with urllib.request.urlopen(request, timeout=60) as response:
            data = json.loads(response.read().decode("utf-8"))
    except urllib.error.HTTPError as error:
        detail = error.read().decode("utf-8")[:500]
        return f"[ERROR] Gemini API {error.code}: {detail}"

    return data["candidates"][0]["content"]["parts"][0]["text"]

## 1. Direct Prompt vs Reasoning Prompt

Bagian ini membandingkan prompt langsung dan prompt reasoning. Jika API key tersedia, dua prompt ini bisa langsung dikirim ke LLM.

In [ ]:
question = "Sebuah toko memberi diskon 20% dari harga 150000, lalu pajak 10% dari harga setelah diskon. Berapa total akhirnya?"

direct_prompt = f"Jawab singkat: {question}"
reasoning_prompt = f"Pecahkan bertahap, cek tiap angka, lalu beri jawaban akhir: {question}"

print("Direct prompt:\n", direct_prompt)
print("\nReasoning prompt:\n", reasoning_prompt)

In [ ]:
print("Direct LLM response:\n")
print(call_gemini_text(direct_prompt))

print("\nReasoning LLM response:\n")
print(call_gemini_text(reasoning_prompt))

In [ ]:
price = 150_000
discounted_price = price * (1 - 0.20)
final_price = discounted_price * (1 + 0.10)

print("Harga awal:", price)
print("Setelah diskon:", int(discounted_price))
print("Setelah pajak:", int(final_price))

## 2. Estimasi Token dan Biaya Konteks

Estimator ini kasar karena token asli bergantung pada tokenizer model. Namun cukup untuk menunjukkan bahwa konteks panjang membawa biaya.

In [ ]:
def rough_token_count(text):
    return max(1, int(len(text.split()) * 1.3))

def estimate_cost(text, price_per_million_tokens):
    tokens = rough_token_count(text)
    cost = tokens / 1_000_000 * price_per_million_tokens
    return tokens, cost

short_context = "Ringkas isi dokumen ini. " * 20
long_context = "Ringkas isi dokumen ini. " * 2_000

for name, text in [("short", short_context), ("long", long_context)]:
    tokens, cost = estimate_cost(text, price_per_million_tokens=1.25)
    print(name, "tokens=", tokens, "estimated_cost_usd=", round(cost, 4))

## 3. Lost in the Middle

Simulasi ini menaruh fakta penting di posisi berbeda. Retriever sederhana berbasis keyword akan mudah menemukan fakta, sedangkan prompt panjang mentah berisiko membuat informasi tengah tenggelam.

In [ ]:
filler = "Paragraf administratif yang tidak penting untuk pertanyaan. "
fact = "FAKTA_PENTING: kode proyek internal adalah ORION-27. "

documents = {
    "fact_at_start": fact + filler * 80,
    "fact_in_middle": filler * 40 + fact + filler * 40,
    "fact_at_end": filler * 80 + fact,
}

for name, text in documents.items():
    fact_position = text.index("FAKTA_PENTING") / len(text)
    print(name, "relative_position=", round(fact_position, 2), "rough_tokens=", rough_token_count(text))

In [ ]:
names = list(documents.keys())
positions = [documents[name].index("FAKTA_PENTING") / len(documents[name]) for name in names]

plt.figure(figsize=(7, 2.5))
plt.barh(names, positions)
plt.xlim(0, 1)
plt.xlabel("Posisi relatif fakta penting")
plt.title("Fakta bisa berada di awal, tengah, atau akhir konteks")
plt.tight_layout()
plt.show()

## 4. Retrieval Kecil untuk Mengurangi Konteks

RAG mengambil bagian relevan dulu, sehingga model tidak selalu perlu membaca semua dokumen panjang.

In [ ]:
chunks = [
    "Kebijakan cuti tahunan berlaku untuk semua pegawai tetap.",
    "Kode proyek internal adalah ORION-27 dan hanya dipakai tim riset.",
    "Rapat bulanan dilakukan pada minggu pertama setiap bulan.",
    "Dokumen keuangan disimpan oleh tim akuntansi.",
]
query = "Apa kode proyek internal tim riset?"

def keyword_score(query, chunk):
    query_terms = Counter(query.lower().replace("?", "").split())
    chunk_terms = Counter(chunk.lower().replace(".", "").split())
    return sum(min(query_terms[word], chunk_terms[word]) for word in query_terms)

ranked_chunks = sorted(chunks, key=lambda chunk: keyword_score(query, chunk), reverse=True)
print("Query:", query)
print("Chunk teratas:", ranked_chunks[0])

## 5. Jawaban LLM Berbasis Konteks

Cell ini menunjukkan pola long context vs RAG: hanya chunk teratas yang dimasukkan sebagai konteks.

In [ ]:
rag_prompt = f"""
Jawab pertanyaan hanya berdasarkan konteks berikut.
Jika konteks tidak cukup, katakan bahwa informasinya tidak tersedia.

Konteks:
{ranked_chunks[0]}

Pertanyaan:
{query}
"""

print(call_gemini_text(rag_prompt))